# 01_performance_baseline

This notebook establishes a business performance baseline by **summarizing key KPIs, historical trends, and performance across customers, products, sales channels, and operations**. The results provide a reference point for subsequent forecasting and business analysis.

# Setup

In [5]:
# ==========================================
# Visualization Settings
# ==========================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# ---------- Pandas ----------
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

# ---------- Theme ----------
PRIMARY = "#E11D48"      
SECONDARY = "#F43F5E"
ACCENT = "#FB7185"

PALETTE = [
    PRIMARY,
    SECONDARY,
    ACCENT,
    "#EC4899",
    "#BE185D",
    "#FDA4AF",
]

plt.style.use("default")

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "figure.dpi": 120,

    "axes.spines.top": False,
    "axes.spines.right": False,

    "axes.grid": True,
    "grid.alpha": 0.25,

    "axes.prop_cycle": plt.cycler(color=PALETTE),
})

In [7]:
# ==========================================
# Locate Project Directory
# ==========================================

project_root = Path.cwd().resolve()

while not (project_root / "data").exists():
    if project_root.parent == project_root:
        raise FileNotFoundError("Project root not found.")
    project_root = project_root.parent

raw_dir = project_root / "data" / "raw"

print(f"Project root : {project_root}")
print(f"Raw directory: {raw_dir}")

Project root : /home/pamern/Projects/fashion-ecommerce-analytics
Raw directory: /home/pamern/Projects/fashion-ecommerce-analytics/data/raw


In [8]:
# ==========================================
# Load Data
# ==========================================

tables = {
    file.stem: pd.read_csv(file)
    for file in raw_dir.glob("*.csv")
}

print(f"Loaded {len(tables)} tables.\n")

for name, df in tables.items():
    print(f"{name:<15} {df.shape[0]:>8,} rows × {df.shape[1]} columns")

Loaded 14 tables.

customers        121,930 rows × 7 columns
geography         39,948 rows × 4 columns
inventory         60,247 rows × 17 columns
order_items      714,669 rows × 7 columns
orders           646,945 rows × 8 columns
payments         646,945 rows × 4 columns
products           2,412 rows × 8 columns
promotions            50 rows × 10 columns
returns           39,939 rows × 7 columns
reviews          113,551 rows × 7 columns
sales              3,833 rows × 3 columns
sample_submission      548 rows × 3 columns
shipments        566,067 rows × 4 columns
web_traffic        3,652 rows × 7 columns


In [9]:
# ==========================================
# Parse Date Columns
# ==========================================

DATE_COLUMNS = {
    "signup_date",
    "order_date",
    "ship_date",
    "delivery_date",
    "review_date",
    "return_date",
    "snapshot_date",
    "start_date",
    "end_date",
}

for df in tables.values():
    for col in DATE_COLUMNS & set(df.columns):
        df[col] = pd.to_datetime(df[col], errors="coerce")

# 1 - Revenue Overview